# FADS region from MDD sumstats

In [2]:
from pyprojroot.here import here
import polars as pl

## dbSNP query

Query region from dbSNP to get SNPs and coordinates. Reference files downloaded from [MRC IEU](https://mrcieu.github.io/gwas2vcf/install/).

In [8]:
!bcftools view \
    --regions 'chr11:61000000-63000000' \
    --output-type u \
    {here()}/data/reference/dbsnp.v153.hg38.vcf.gz |\
bcftools norm --multiallelics -both \
    --output-type u |\
bcftools query --print-header \
    --format '%CHROM\t%POS\t%ID\t%REF\t%ALT\n' > {here()}/data/reference/chr11_61000000_63000000_rsid.tsv

Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	470239/42466/0/0/0/0/0


In [14]:
rsids = pl.read_csv(here("data/reference/chr11_61000000_63000000_rsid.tsv"), separator = "\t")
print(rsids)

shape: (522_156, 5)
┌───────────┬──────────┬──────────────┬────────┬────────┐
│ #[1]CHROM ┆ [2]POS   ┆ [3]ID        ┆ [4]REF ┆ [5]ALT │
│ ---       ┆ ---      ┆ ---          ┆ ---    ┆ ---    │
│ str       ┆ i64      ┆ str          ┆ str    ┆ str    │
╞═══════════╪══════════╪══════════════╪════════╪════════╡
│ chr11     ┆ 61000001 ┆ rs1461210290 ┆ G      ┆ A      │
│ chr11     ┆ 61000003 ┆ rs994318808  ┆ G      ┆ T      │
│ chr11     ┆ 61000006 ┆ rs1352436398 ┆ C      ┆ A      │
│ chr11     ┆ 61000007 ┆ rs577507158  ┆ G      ┆ A      │
│ chr11     ┆ 61000013 ┆ rs145003773  ┆ C      ┆ T      │
│ …         ┆ …        ┆ …            ┆ …      ┆ …      │
│ chr11     ┆ 62999982 ┆ rs1002753088 ┆ A      ┆ T      │
│ chr11     ┆ 62999983 ┆ rs1048751966 ┆ C      ┆ G      │
│ chr11     ┆ 62999986 ┆ rs1452838713 ┆ C      ┆ A      │
│ chr11     ┆ 62999987 ┆ rs1197886568 ┆ T      ┆ C      │
│ chr11     ┆ 62999998 ┆ rs763377344  ┆ T      ┆ C      │
└───────────┴──────────┴──────────────┴────────┴────

## MDD sumstats

MDD2025 (Adams et al.)

In [32]:
mdd2025_eur = pl.read_csv(here("data/raw/sumstats/daner_pgc_mdd_full_eur_hg19_v3.49.24.11.neff.gz"), separator = "\t")

MDD2024 (Meng et al.)

In [47]:
mdd2024_afr = pl.read_csv(
    here("data/raw/sumstats/mdd2023diverse_AFR_Neff.csv"),
    null_values = ["NA"],
    ignore_errors = True
).with_columns(
    pl.lit("AFR").alias("cluster")
)

mdd2024_eas = pl.read_csv(
    here("data/raw/sumstats/mdd2023diverse_EAS_Neff.csv"),
    null_values = ["NA"],
    ignore_errors = True
).with_columns(
    pl.lit("EAS").alias("cluster")
)

mdd2024_his = pl.read_csv(
    here("data/raw/sumstats/mdd2023diverse_HIS_Neff.csv"),
    null_values = ["NA"],
    ignore_errors = True
).with_columns(
    pl.lit("HIS").alias("cluster")
)

mdd2024_sas = pl.read_csv(
    here("data/raw/sumstats/mdd2023diverse_SAS_Neff.csv"),
    null_values = ["NA"],
    ignore_errors = True
).with_columns(
    pl.lit("SAS").alias("cluster")
)

mdd2024 = pl.concat(
    [
        mdd2024_afr,
        mdd2024_eas,
        mdd2024_his,
        mdd2024_sas,
    ],
    how = "vertical"
)


## Merge and update positions

In [36]:
mdd2025_eur_lifted = (mdd2025_eur
    .join(rsids,
          left_on = "SNP",
          right_on = "[3]ID",
          how = "inner"
    )
    .filter(
        ((pl.col("A1") == pl.col("[4]REF")) & (pl.col("A2") == pl.col("[5]ALT"))) |
        ((pl.col("A1") == pl.col("[5]ALT")) & (pl.col("A2") == pl.col("[4]REF")))
    )
    .select(
        pl.col("CHR"),
        pl.col("[2]POS").alias("BP"),
        pl.col("SNP"),
        pl.col("A1"),
        pl.col("A2"),
        pl.col("OR"),
        pl.col("SE"),
        pl.col("P").alias("PVAL")
    )
)
mdd2025_eur_lifted.write_csv(here("data/processed/sumstats/mdd2025_eur_hg38_chr11_61000000_63000000.tsv.gz"), separator = "\t")

In [51]:
mdd2024_lifted = (mdd2024
    .with_columns(
        pl.col("EA").str.to_uppercase().alias("A1"),
        pl.col("NEA").str.to_uppercase().alias("A2")
    )
    .join(rsids,
          left_on = "SNP",
          right_on = "[3]ID",
          how = "inner"
    )
    .filter(
        ((pl.col("A1") == pl.col("[4]REF")) & (pl.col("A2") == pl.col("[5]ALT"))) |
        ((pl.col("A1") == pl.col("[5]ALT")) & (pl.col("A2") == pl.col("[4]REF")))
    )
    .select(
        pl.col("Chromosome").alias("CHR"),
        pl.col("[2]POS").alias("BP"),
        pl.col("SNP"),
        pl.col("A1"),
        pl.col("A2"),
        pl.col("logOR").exp().alias("OR"),
        pl.col("SE"),
        pl.col("P").alias("PVAL"),
        pl.col("cluster")
    )
)

In [ ]:
mdd2024_afr_lifted = mdd2024_lifted.filter(
    pl.col("cluster") == "AFR"
).drop(
    "cluster"
)

mdd2024_eas_lifted = mdd2024_lifted.filter(
    pl.col("cluster") == "EAS"
).drop(
    "cluster"
)

mdd2024_his_lifted = mdd2024_lifted.filter(
    pl.col("cluster") == "HIS"
).drop(
    "cluster"
)

mdd2024_sas_lifted = mdd2024_lifted.filter(
    pl.col("cluster") == "SAS"
).drop(
    "cluster"
)

mdd2024_afr_lifted.write_csv(here("data/processed/sumstats/mdd2024_afr_hg38_chr11_61000000_63000000.tsv.gz"), separator = "\t")
mdd2024_eas_lifted.write_csv(here("data/processed/sumstats/mdd2024_eas_hg38_chr11_61000000_63000000.tsv.gz"), separator = "\t")
mdd2024_his_lifted.write_csv(here("data/processed/sumstats/mdd2024_his_hg38_chr11_61000000_63000000.tsv.gz"), separator = "\t")
mdd2024_sas_lifted.write_csv(here("data/processed/sumstats/mdd2024_sas_hg38_chr11_61000000_63000000.tsv.gz"), separator = "\t")